# 02 Baseline Strategies: Static Band + TWAP/VWAP

This notebook is the first empirical hedging layer after the simulated executed-trade tape. It follows the updated project plan in `docs/`:

- Convert executed option trades into signed delta arrivals.
- Backtest zero-hedge and full-hedge benchmarks.
- Backtest static no-trade bands around zero delta.
- Execute band-triggered hedge orders using TWAP and VWAP schedules.
- Measure execution cost, residual delta risk, and the combined objective.

Primary input: `data/simulated/simulated_executed_trades.csv`.

Primary output: `outputs/tables/baseline_strategy_results.csv`.

## 1. Setup

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
SIM_DIR = DATA_DIR / 'simulated'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUT_TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXECUTED_TRADES_PATH = SIM_DIR / 'simulated_executed_trades.csv'
MARKET_INPUTS_PATH = PROCESSED_DIR / 'market_inputs_2026.csv'

OUTPUT_DELTA_ARRIVALS = PROCESSED_DIR / 'executed_trade_delta_arrivals.csv'
OUTPUT_HEDGE_EVENTS = PROCESSED_DIR / 'baseline_hedge_events.csv'
OUTPUT_RESULTS = OUTPUT_TABLE_DIR / 'baseline_strategy_results.csv'

# $50 of P&L per ES index point per contract. ES options and ES futures share this
# multiplier, so it cancels out of the hedge ratio: contract counts are delta * quantity
# with NO multiplier. It re-enters only when converting points to dollars (notional,
# execution cost, risk penalty).
ES_MULTIPLIER = 50
BAR_MINUTES = 5
BAR_YEAR_FRACTION = BAR_MINUTES / (252 * 6.5 * 60)

# No-trade bands on net book delta, in ES futures contracts.
BAND_GRID = [500, 1_000, 2_000, 4_000]
HORIZON_GRID_MINUTES = [15, 30, 60]
EXECUTION_METHODS = ['twap', 'vwap']

SPREAD_BPS = 1.0
IMPACT_ALPHA = 0.05
IMPACT_BETA = 0.5
LAMBDA_RISK = 1e-8

print(PROJECT_ROOT)

C:\Users\chung\Downloads\JPM delta hedging project\JPM_Optimal_Intraday_Delta_Hedging_Strategy


## 2. Load Executed Trades and Market Inputs

In [2]:
def load_executed_trades(path: Path) -> pd.DataFrame:
    trades = pd.read_csv(path)
    trades['timestamp_utc'] = pd.to_datetime(trades['timestamp'], utc=True)
    trades['timestamp_et'] = trades['timestamp_utc'].dt.tz_convert('America/New_York').dt.tz_localize(None)
    trades['trade_date'] = trades['timestamp_et'].dt.normalize()
    trades['expiry_date'] = pd.to_datetime(trades['expiry_date'])
    trades['side'] = trades['side'].str.upper()
    trades['opt_type'] = trades['opt_type'].str.upper()
    return trades.sort_values('timestamp_et').reset_index(drop=True)


def load_market_inputs(path: Path) -> pd.DataFrame:
    market = pd.read_csv(path)
    market['timestamp_et'] = pd.to_datetime(market['timestamp_et'])
    market['trade_date'] = pd.to_datetime(market['trade_date'])
    return market.sort_values('timestamp_et').reset_index(drop=True)


trades_raw = load_executed_trades(EXECUTED_TRADES_PATH)
market = load_market_inputs(MARKET_INPUTS_PATH)

print(f'Executed trades: {len(trades_raw):,}')
print(f'Market bars:     {len(market):,}')
trades_raw.head()

Executed trades: 32,127
Market bars:     8,364


,trade_id,timestamp,sec_of_day,symbol,root,expiry_code,expiry_date,opt_type,strike,side,quantity,bid,ask,exec_price,timestamp_utc,timestamp_et,trade_date
0,1,2026-01-02 14:42:39+00:00,52959,ESM6 P6100,ES,ESM6,2026-06-19,P,6100.0,SELL,2,95.75,96.00,95.345960,2026-01-02 14:42:39+00:00,2026-01-02 09:42:39,2026-01-02
1,2,2026-01-02 14:48:58+00:00,53338,ESH6 C7325,ES,ESH6,2026-03-20,C,7325.0,SELL,15,19.75,20.00,20.209821,2026-01-02 14:48:58+00:00,2026-01-02 09:48:58,2026-01-02
2,3,2026-01-02 14:52:13+00:00,53533,ESM6 P6800,ES,ESM6,2026-06-19,P,6800.0,BUY,15,225.75,226.25,220.270833,2026-01-02 14:52:13+00:00,2026-01-02 09:52:13,2026-01-02
3,4,2026-01-02 14:53:41+00:00,53621,ESH6 C7000,ES,ESH6,2026-03-20,C,7000.0,BUY,1,104.50,104.75,112.590909,2026-01-02 14:53:41+00:00,2026-01-02 09:53:41,2026-01-02
4,5,2026-01-02 14:53:57+00:00,53637,ESH6 P4500,ES,ESH6,2026-03-20,P,4500.0,BUY,5,4.45,4.50,4.632456,2026-01-02 14:53:57+00:00,2026-01-02 09:53:57,2026-01-02


## 3. Align Trades to 5-Minute Market Bars

The execution tape is timestamped at trade time. The backtest works on the same 5-minute grid as the market inputs, so each trade is assigned to the latest available market bar.

In [3]:
def align_trades_to_market(trades: pd.DataFrame, market: pd.DataFrame) -> pd.DataFrame:
    aligned = pd.merge_asof(
        trades.sort_values('timestamp_et'),
        market.sort_values('timestamp_et'),
        on='timestamp_et',
        direction='backward',
        tolerance=pd.Timedelta(minutes=10),
        suffixes=('', '_mkt'),
    )
    missing = aligned['es1_price'].isna().sum()
    if missing:
        print(f'Dropping {missing:,} trades that could not be aligned to a market bar.')
        aligned = aligned.dropna(subset=['es1_price']).copy()
    aligned['bar_time'] = aligned['timestamp_et'].dt.floor(f'{BAR_MINUTES}min')
    return aligned.reset_index(drop=True)


trades_aligned = align_trades_to_market(trades_raw, market)
trades_aligned[['trade_id', 'timestamp_et', 'bar_time', 'symbol', 'opt_type', 'strike', 'quantity', 'es1_price', 'bvol_1m']].head()

,trade_id,timestamp_et,bar_time,symbol,opt_type,strike,quantity,es1_price,bvol_1m
0,1,2026-01-02 09:42:39,2026-01-02 09:40:00,ESM6 P6100,P,6100.0,2,6919.75,0.12025
1,2,2026-01-02 09:48:58,2026-01-02 09:45:00,ESH6 C7325,C,7325.0,15,6931.75,0.12025
2,3,2026-01-02 09:52:13,2026-01-02 09:50:00,ESM6 P6800,P,6800.0,15,6932.75,0.12025
3,4,2026-01-02 09:53:41,2026-01-02 09:50:00,ESH6 C7000,C,7000.0,1,6932.75,0.12025
4,5,2026-01-02 09:53:57,2026-01-02 09:50:00,ESH6 P4500,P,4500.0,5,6932.75,0.12025


## 4. Convert Option Executions to Delta Arrivals

The executed-trade tape is ES option flow, so the framework uses a Black-76-style futures-option delta. This cell creates the signed book-delta arrival from each option execution.

Units matter here: one ES option delivers one ES futures contract, and both carry the same $50-per-index-point multiplier, so the multiplier cancels out of the hedge ratio. `signed_delta_contracts = delta * quantity` is therefore already the number of ES futures contracts needed to offset the trade; the $50 multiplier is applied only when converting to dollar notional (`signed_delta_notional`) and dollar costs.

In [4]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def black76_delta(futures_price: float, strike: float, years: float, vol: float, option_type: str, rate: float = 0.0) -> float:
    years = max(float(years), 1 / 365)
    vol = max(float(vol), 0.01)
    futures_price = max(float(futures_price), 0.01)
    strike = max(float(strike), 0.01)
    discount = math.exp(-float(rate) * years)
    d1 = (math.log(futures_price / strike) + 0.5 * vol**2 * years) / (vol * math.sqrt(years))
    if option_type == 'C':
        return discount * norm_cdf(d1)
    return -discount * norm_cdf(-d1)


def add_delta_arrivals(trades: pd.DataFrame) -> pd.DataFrame:
    out = trades.copy()
    out['dte'] = (out['expiry_date'] - out['trade_date']).dt.days.clip(lower=1)
    out['years_to_expiry'] = out['dte'] / 365.0
    out['delta'] = [
        black76_delta(row.es1_price, row.strike, row.years_to_expiry, row.bvol_1m, row.opt_type, row.sofr_1m_rate)
        for row in out.itertuples(index=False)
    ]
    out['side_sign'] = np.where(out['side'].eq('BUY'), 1.0, -1.0)
    # One ES option delivers one ES future, so the hedge ratio is delta * quantity in
    # futures contracts -- the shared $50 multiplier cancels and must NOT appear here.
    out['signed_delta_contracts'] = out['side_sign'] * out['delta'] * out['quantity']
    out['signed_delta_notional'] = out['signed_delta_contracts'] * ES_MULTIPLIER * out['es1_price']
    return out


delta_arrivals = add_delta_arrivals(trades_aligned)
delta_arrivals.to_csv(OUTPUT_DELTA_ARRIVALS, index=False)

delta_arrivals[['trade_id', 'timestamp_et', 'symbol', 'side', 'quantity', 'delta', 'signed_delta_contracts', 'signed_delta_notional']].head()

,trade_id,timestamp_et,symbol,side,quantity,delta,signed_delta_contracts,signed_delta_notional
0,1,2026-01-02 09:42:39,ESM6 P6100,SELL,2,-5.538310e-02,1.107662e-01,3.832372e+04
1,2,2026-01-02 09:48:58,ESH6 C7325,SELL,15,1.643766e-01,-2.465649e+00,-8.545632e+05
2,3,2026-01-02 09:52:13,ESM6 P6800,BUY,15,-3.840383e-01,-5.760575e+00,-1.996831e+06
3,4,2026-01-02 09:53:41,ESH6 C7000,BUY,1,4.380888e-01,4.380888e-01,1.518580e+05
4,5,2026-01-02 09:53:57,ESH6 P4500,BUY,5,-2.038048e-15,-1.019024e-14,-3.532319e-09


## 5. Build the Backtest Grid

The event grid combines market bars with delta arrivals. The strategy simulator will update book delta at each bar, apply scheduled child hedge orders, then decide whether a new hedge schedule should replace the current one.

In [5]:
def build_delta_bar_grid(delta_arrivals: pd.DataFrame, market: pd.DataFrame) -> pd.DataFrame:
    arrivals = (
        delta_arrivals.groupby('bar_time', as_index=False)
        .agg(
            option_delta_arrival=('signed_delta_contracts', 'sum'),
            option_delta_notional=('signed_delta_notional', 'sum'),
            n_option_trades=('trade_id', 'count'),
        )
    )
    grid = market.rename(columns={'timestamp_et': 'bar_time'}).copy()
    grid = grid.merge(arrivals, on='bar_time', how='left')
    grid[['option_delta_arrival', 'option_delta_notional', 'n_option_trades']] = grid[
        ['option_delta_arrival', 'option_delta_notional', 'n_option_trades']
    ].fillna(0.0)
    grid['trade_date'] = pd.to_datetime(grid['trade_date'])
    return grid.sort_values('bar_time').reset_index(drop=True)


bar_grid = build_delta_bar_grid(delta_arrivals, market)
bar_grid[['bar_time', 'trade_date', 'es1_price', 'es1_volume_5m', 'option_delta_arrival', 'n_option_trades']].head()

,bar_time,trade_date,es1_price,es1_volume_5m,option_delta_arrival,n_option_trades
0,2026-01-02 09:30:00,2026-01-02,6918.75,40615.0,0.000000,0.0
1,2026-01-02 09:35:00,2026-01-02,6915.25,24586.0,0.000000,0.0
2,2026-01-02 09:40:00,2026-01-02,6919.75,20690.0,0.110766,1.0
3,2026-01-02 09:45:00,2026-01-02,6931.75,33464.0,-2.465649,1.0
4,2026-01-02 09:50:00,2026-01-02,6932.75,25026.0,-2.847150,4.0


## 6. Execution Cost and Scheduling Helpers

Hedge orders are ES futures, which trade only in whole contracts. Child schedules are still computed as fractional slices (a TWAP split of a parent order rarely lands on integers), but execution accumulates the fractional slices in a carry and fills only the whole-contract part each bar; the remainder rolls forward until it rounds to a full contract. Costs are charged in dollars on the integer quantity actually traded.

In [6]:
def estimate_child_order_cost(hedge_contracts: float, price: float, adv: float) -> tuple[float, float, float]:
    """Return total, spread, and impact cost in dollars for an ES child hedge order.

    `hedge_contracts` and `adv` are both in ES futures contracts, so participation is a
    true contract-on-contract rate; the $50 multiplier converts point costs to dollars.
    """
    qty = abs(float(hedge_contracts))
    price = float(price)
    adv = max(float(adv), 1.0)
    half_spread_points = price * (SPREAD_BPS / 10_000) / 2
    spread_cost = qty * half_spread_points * ES_MULTIPLIER
    participation = qty / adv
    impact_points = IMPACT_ALPHA * price * (participation ** IMPACT_BETA)
    impact_cost = qty * impact_points * ES_MULTIPLIER
    return spread_cost + impact_cost, spread_cost, impact_cost


def make_child_schedule(parent_delta: float, start_idx: int, day_grid: pd.DataFrame, horizon_minutes: int, method: str) -> pd.DataFrame:
    n_child = max(int(horizon_minutes / BAR_MINUTES), 1)
    idx = list(range(start_idx, min(start_idx + n_child, len(day_grid))))
    if not idx:
        return pd.DataFrame(columns=['idx', 'hedge_delta'])

    if method == 'twap':
        weights = np.ones(len(idx), dtype=float) / len(idx)
    elif method == 'vwap':
        volume = day_grid.loc[idx, 'es1_volume_5m'].clip(lower=0).fillna(0).to_numpy(dtype=float)
        weights = volume / volume.sum() if volume.sum() > 0 else np.ones(len(idx), dtype=float) / len(idx)
    else:
        raise ValueError(f'Unknown execution method: {method}')

    return pd.DataFrame({'idx': idx, 'hedge_delta': parent_delta * weights})

## 7. Strategy Simulator Framework

In [7]:
def simulate_static_band_day(day_grid: pd.DataFrame, band: float, method: str, horizon_minutes: int) -> tuple[dict, pd.DataFrame]:
    book_delta = 0.0
    active_schedule = pd.DataFrame(columns=['idx', 'hedge_delta'])
    # ES futures trade only in whole contracts. Scheduled child slices are fractional,
    # so unexecuted remainders accumulate here and fill once they round to >= 1 contract.
    pending_hedge = 0.0
    hedge_events = []
    risk_cost = 0.0
    exec_cost = 0.0
    spread_cost = 0.0
    impact_cost = 0.0
    total_abs_hedge = 0.0
    triggers = 0

    day_grid = day_grid.reset_index(drop=True).copy()

    for i, row in day_grid.iterrows():
        book_delta += float(row.option_delta_arrival)

        scheduled = active_schedule.loc[active_schedule['idx'].eq(i), 'hedge_delta']
        pending_hedge += float(scheduled.sum()) if not scheduled.empty else 0.0
        hedge_contracts = int(round(pending_hedge))
        if hedge_contracts != 0:
            pending_hedge -= hedge_contracts
            cost, spread, impact = estimate_child_order_cost(hedge_contracts, row.es1_price, row.es1_adv_20d)
            exec_cost += cost
            spread_cost += spread
            impact_cost += impact
            total_abs_hedge += abs(hedge_contracts)
            book_delta += hedge_contracts
            hedge_events.append({
                'bar_time': row.bar_time,
                'event_type': 'child_order',
                'band': band,
                'method': method,
                'horizon_minutes': horizon_minutes,
                'hedge_delta': hedge_contracts,
                'book_delta_after': book_delta,
                'exec_cost': cost,
                'spread_cost': spread,
                'impact_cost': impact,
            })

        dollar_delta_per_point = book_delta * ES_MULTIPLIER
        risk_cost += (dollar_delta_per_point ** 2) * (float(row.bvol_1m) ** 2) * BAR_YEAR_FRACTION

        if abs(book_delta) > band:
            triggers += 1
            parent_order = -book_delta
            pending_hedge = 0.0  # stale carry is superseded along with the old schedule
            active_schedule = make_child_schedule(parent_order, i + 1, day_grid, horizon_minutes, method)
            hedge_events.append({
                'bar_time': row.bar_time,
                'event_type': 'trigger',
                'band': band,
                'method': method,
                'horizon_minutes': horizon_minutes,
                'hedge_delta': parent_order,
                'book_delta_after': book_delta,
                'exec_cost': 0.0,
                'spread_cost': 0.0,
                'impact_cost': 0.0,
            })

    eod_delta_before_flatten = book_delta
    close_contracts = -int(round(book_delta))
    if close_contracts != 0:
        row = day_grid.iloc[-1]
        cost, spread, impact = estimate_child_order_cost(close_contracts, row.es1_price, row.es1_adv_20d)
        exec_cost += cost
        spread_cost += spread
        impact_cost += impact
        total_abs_hedge += abs(close_contracts)
        book_delta += close_contracts
        hedge_events.append({
            'bar_time': row.bar_time,
            'event_type': 'close_flatten',
            'band': band,
            'method': method,
            'horizon_minutes': horizon_minutes,
            'hedge_delta': close_contracts,
            'book_delta_after': book_delta,
            'exec_cost': cost,
            'spread_cost': spread,
            'impact_cost': impact,
        })

    result = {
        'trade_date': day_grid['trade_date'].iloc[0],
        'strategy': 'static_band',
        'band': band,
        'method': method,
        'horizon_minutes': horizon_minutes,
        'execution_cost': exec_cost,
        'spread_cost': spread_cost,
        'impact_cost': impact_cost,
        'delta_risk': risk_cost,
        'combined_loss': exec_cost + LAMBDA_RISK * risk_cost,
        'n_triggers': triggers,
        'total_abs_hedge_delta': total_abs_hedge,
        'eod_delta_before_flatten': eod_delta_before_flatten,
    }
    return result, pd.DataFrame(hedge_events)

## 8. Benchmark Frameworks

In [8]:
def simulate_zero_hedge_day(day_grid: pd.DataFrame) -> dict:
    book_delta = 0.0
    risk_cost = 0.0
    day_grid = day_grid.reset_index(drop=True)
    for row in day_grid.itertuples(index=False):
        book_delta += float(row.option_delta_arrival)
        dollar_delta_per_point = book_delta * ES_MULTIPLIER
        risk_cost += (dollar_delta_per_point ** 2) * (float(row.bvol_1m) ** 2) * BAR_YEAR_FRACTION
    close_row = day_grid.iloc[-1]
    close_contracts = -int(round(book_delta))
    exec_cost, spread_cost, impact_cost = estimate_child_order_cost(close_contracts, close_row.es1_price, close_row.es1_adv_20d)
    return {
        'trade_date': day_grid['trade_date'].iloc[0],
        'strategy': 'zero_hedge',
        'band': np.nan,
        'method': 'close_only',
        'horizon_minutes': 0,
        'execution_cost': exec_cost,
        'spread_cost': spread_cost,
        'impact_cost': impact_cost,
        'delta_risk': risk_cost,
        'combined_loss': exec_cost + LAMBDA_RISK * risk_cost,
        'n_triggers': 0,
        'total_abs_hedge_delta': abs(close_contracts),
        'eod_delta_before_flatten': book_delta,
    }


def simulate_full_hedge_day(day_grid: pd.DataFrame) -> dict:
    book_delta = 0.0
    risk_cost = 0.0
    exec_cost = 0.0
    spread_cost = 0.0
    impact_cost = 0.0
    total_abs_hedge = 0.0
    n_triggers = 0
    day_grid = day_grid.reset_index(drop=True)
    for row in day_grid.itertuples(index=False):
        book_delta += float(row.option_delta_arrival)
        # Hedge to the nearest whole contract; the sub-contract remainder cannot be
        # traded and stays on the book until later arrivals tip it over half a contract.
        hedge_contracts = -int(round(book_delta))
        if hedge_contracts != 0:
            cost, spread, impact = estimate_child_order_cost(hedge_contracts, row.es1_price, row.es1_adv_20d)
            exec_cost += cost
            spread_cost += spread
            impact_cost += impact
            total_abs_hedge += abs(hedge_contracts)
            book_delta += hedge_contracts
            n_triggers += 1
        dollar_delta_per_point = book_delta * ES_MULTIPLIER
        risk_cost += (dollar_delta_per_point ** 2) * (float(row.bvol_1m) ** 2) * BAR_YEAR_FRACTION
    return {
        'trade_date': day_grid['trade_date'].iloc[0],
        'strategy': 'full_hedge',
        'band': 0.0,
        'method': 'immediate',
        'horizon_minutes': 0,
        'execution_cost': exec_cost,
        'spread_cost': spread_cost,
        'impact_cost': impact_cost,
        'delta_risk': risk_cost,
        'combined_loss': exec_cost + LAMBDA_RISK * risk_cost,
        'n_triggers': n_triggers,
        'total_abs_hedge_delta': total_abs_hedge,
        'eod_delta_before_flatten': book_delta,
    }

## 9. Run the Baseline Grid

In [9]:
results = []
hedge_logs = []

for trade_date, day_grid in bar_grid.groupby('trade_date'):
    day_grid = day_grid.sort_values('bar_time').reset_index(drop=True)
    results.append(simulate_zero_hedge_day(day_grid))
    results.append(simulate_full_hedge_day(day_grid))

    for band in BAND_GRID:
        for method in EXECUTION_METHODS:
            for horizon in HORIZON_GRID_MINUTES:
                day_result, day_hedges = simulate_static_band_day(day_grid, band, method, horizon)
                results.append(day_result)
                if not day_hedges.empty:
                    day_hedges['trade_date'] = trade_date
                    hedge_logs.append(day_hedges)

results_df = pd.DataFrame(results)
hedge_events = pd.concat(hedge_logs, ignore_index=True) if hedge_logs else pd.DataFrame()

results_df.to_csv(OUTPUT_RESULTS, index=False)
hedge_events.to_csv(OUTPUT_HEDGE_EVENTS, index=False)

print(f'Wrote results to {OUTPUT_RESULTS.relative_to(PROJECT_ROOT)}')
print(f'Wrote hedge events to {OUTPUT_HEDGE_EVENTS.relative_to(PROJECT_ROOT)}')
results_df.head()

Wrote results to outputs\tables\baseline_strategy_results.csv
Wrote hedge events to data\processed\baseline_hedge_events.csv


,trade_date,strategy,band,method,horizon_minutes,execution_cost,spread_cost,impact_cost,delta_risk,combined_loss,n_triggers,total_abs_hedge_delta,eod_delta_before_flatten
0,2026-01-02,zero_hedge,NaN,close_only,0,10527.530387,414.135000,10113.395387,73.101563,10527.530388,0,24.0,24.144509
1,2026-01-02,full_hedge,0.0,immediate,0,40417.268617,2655.366875,37761.901742,0.011575,40417.268617,33,154.0,0.144509
2,2026-01-02,static_band,500.0,twap,15,10527.530387,414.135000,10113.395387,73.101563,10527.530388,0,24.0,24.144509
3,2026-01-02,static_band,500.0,twap,30,10527.530387,414.135000,10113.395387,73.101563,10527.530388,0,24.0,24.144509
4,2026-01-02,static_band,500.0,twap,60,10527.530387,414.135000,10113.395387,73.101563,10527.530388,0,24.0,24.144509


## 10. Summarize Strategy Performance

In [10]:
summary = (
    results_df.groupby(['strategy', 'band', 'method', 'horizon_minutes'], dropna=False)
    .agg(
        execution_cost=('execution_cost', 'sum'),
        delta_risk=('delta_risk', 'sum'),
        combined_loss=('combined_loss', 'sum'),
        n_triggers=('n_triggers', 'sum'),
        total_abs_hedge_delta=('total_abs_hedge_delta', 'sum'),
        avg_eod_delta_before_flatten=('eod_delta_before_flatten', 'mean'),
    )
    .reset_index()
    .sort_values('combined_loss')
)

summary.head(15)

,strategy,band,method,horizon_minutes,execution_cost,delta_risk,combined_loss,n_triggers,total_abs_hedge_delta,avg_eod_delta_before_flatten
3,static_band,500.0,twap,60,1.383818e+06,181774.268692,1.383818e+06,3,7604.0,5.016767
6,static_band,500.0,vwap,60,1.387161e+06,180763.131784,1.387161e+06,2,7604.0,5.016767
2,static_band,500.0,twap,30,1.413974e+06,180125.236293,1.413974e+06,1,7604.0,5.928532
5,static_band,500.0,vwap,30,1.416372e+06,180225.598692,1.416372e+06,1,7604.0,5.928532
1,static_band,500.0,twap,15,1.446896e+06,178687.783457,1.446896e+06,1,7604.0,5.928532
4,static_band,500.0,vwap,15,1.448239e+06,178532.593889,1.448239e+06,1,7604.0,5.928532
7,static_band,1000.0,twap,15,1.573752e+06,251665.797568,1.573752e+06,0,7604.0,11.036375
8,static_band,1000.0,twap,30,1.573752e+06,251665.797568,1.573752e+06,0,7604.0,11.036375
12,static_band,1000.0,vwap,60,1.573752e+06,251665.797568,1.573752e+06,0,7604.0,11.036375
9,static_band,1000.0,twap,60,1.573752e+06,251665.797568,1.573752e+06,0,7604.0,11.036375


## 11. Next Notebook Handoff

Notebook 03 should consume `baseline_strategy_results.csv` and build:

- efficient-frontier plots in execution-cost vs delta-risk space;
- benchmark comparisons versus zero hedge and full hedge;
- sensitivity tables for band width, execution horizon, impact parameters, and risk penalty;
- the final recommendation for the baseline static-band strategy.